Единственная задача: обобщить написанный на семинаре код и сымитировать работу одного большого отдела ABBYY, в котором есть три маленьких подотдела с лингвистами, программистами и комплингом. То есть, что у нас должно быть реализовано:

- родительский класс "работник"
- базовые классы "лингвист", "программист" и "компьютерный лингвист"
- у всех методы work
- классы "босс_лингвист", "босс_программист" и "босс_кл", которые могут наследовать (с подмешиванием) от общего класса "босс"
- у боссов в атрибутах сидят их подчиненные
- босс подотдела получает квесты от менеджера главного отдела и принуждает сотрудников работать
- в главном отделе есть метод для выдачи квестов
- соответственно, используем как наследование, так и композицию с делегированием

In [1]:
class Employee:
    def __init__(self, name, surname):
        self.name = name
        self.surname = surname
        self.salary = 300
        self.bank = 0

    def work(self):
        raise NotImplementedError

    def __repr__(self):
        return f"{self.__class__.__name__}('{self.name}', '{self.surname}')"

In [2]:
class Linguist(Employee):
    def __init__(self, name, surname):
        super().__init__(name, surname)
        self.salary *= 1.3
        self.publications = []

    def work(self, hours, task):
        print(f"{self.name} работает над заданием...")
        self.bank += hours * self.salary
        self.publications.append(task)
        print(f"{self.name} сделал публикацию: {task}")

In [3]:
class Programmer(Employee):
    def __init__(self, name, surname):
        super().__init__(name, surname)
        self.salary *= 1.8
        self.projects = []

    def work(self, hours, task):
        print(f"{self.name} пишет код...")
        self.bank += hours * self.salary
        self.projects.append(task)
        print(f"{self.name} закоммитил проект: {task}")

In [4]:
class ComputerLinguist(Linguist, Programmer):
    def __init__(self, name, surname):
        Linguist.__init__(self, name, surname)
        Programmer.__init__(self, name, surname)
        self.salary *= 0.9

    def work(self, hours, task):
        print(f"{self.name} рисует деревья и обучает нейронки одновременно...")
        self.bank += hours * self.salary
        self.publications.append(task)
        self.projects.append(task)
        print(f"{self.name}: дерево + нейронка готовы ({task})")

In [5]:
class Boss:
    def __init__(self, name, surname, subordinates=None):
        self.name = name
        self.surname = surname
        self.subordinates = subordinates if subordinates else []

    def add_employee(self, worker):
        self.subordinates.append(worker)

    def command(self, hours, task):
        print(f"\nбосс {self.name} раздаёт задание '{task}' работникам:")
        for worker in self.subordinates:
            worker.work(hours, task)


class LinguistBoss(Boss, Linguist):
    def __init__(self, name, surname, subordinates=None):
        Boss.__init__(self, name, surname, subordinates)
        Linguist.__init__(self, name, surname)


class ProgrammerBoss(Boss, Programmer):
    def __init__(self, name, surname, subordinates=None):
        Boss.__init__(self, name, surname, subordinates)
        Programmer.__init__(self, name, surname)


class CLBoss(Boss, ComputerLinguist):
    def __init__(self, name, surname, subordinates=None):
        Boss.__init__(self, name, surname, subordinates)
        ComputerLinguist.__init__(self, name, surname)

In [6]:
class Abbyy:
    def __init__(self, *bosses):
        self.bosses = list(bosses)

    def give_quest(self, boss: Boss, task: str, hours: int):
        print(f"\nприемная выдаёт квест: '{task}'")
        boss.command(hours, task)

In [10]:
# я попросила нейронку придумать имена, повеселило:

l1 = Linguist("ира", "лингвистова")
l2 = Linguist("аня", "корпусова")
p1 = Programmer("максим", "байт")
c1 = ComputerLinguist("катя", "нлпшникова")

boss_ling = LinguistBoss("дарья", "фонологова", [l1, l2])
boss_prog = ProgrammerBoss("саша", "алгоритмов", [p1])
boss_cl = CLBoss("оля", "морфемная", [c1])

dept = Abbyy(boss_ling, boss_prog, boss_cl)

dept.give_quest(boss_ling, "статья в вопросы языкознания", 4)
dept.give_quest(boss_prog, "сделать API", 3)
dept.give_quest(boss_cl, "написать диплом про синтаксическую разметку нейронкой", 5)



приемная выдаёт квест: 'статья в вопросы языкознания'

босс дарья раздаёт задание 'статья в вопросы языкознания' работникам:
ира работает над заданием...
ира сделал публикацию: статья в вопросы языкознания
аня работает над заданием...
аня сделал публикацию: статья в вопросы языкознания

приемная выдаёт квест: 'сделать API'

босс саша раздаёт задание 'сделать API' работникам:
максим пишет код...
максим закоммитил проект: сделать API

приемная выдаёт квест: 'написать диплом про синтаксическую разметку нейронкой'

босс оля раздаёт задание 'написать диплом про синтаксическую разметку нейронкой' работникам:
катя рисует деревья и обучает нейронки одновременно...
катя: дерево + нейронка готовы (написать диплом про синтаксическую разметку нейронкой)
